# Configuration Notebook
Central place for all storage paths, credential names, and catalog/schema identifiers.
All other notebooks `%run` this notebook to pick up these Widget-backed parameters.
Override any Widget default at runtime via the Databricks Job UI or `dbutils.widgets.text()`.

In [0]:
# ---------------------------------------------------------------------------
# Widget declarations — override at runtime without changing code.
#
# SECURITY NOTE: In production, the storage credential name should be
# retrieved from Databricks Secrets, not stored as a widget default:
#
#   STORAGE_CREDENTIAL = dbutils.secrets.get(scope="finance-scope",
#                                             key="storage-credential-name")
#
# The widget below is kept for local development / interactive convenience.
# ---------------------------------------------------------------------------
dbutils.widgets.text("storage_account",   "dbstoragefinance",           "Storage Account")
dbutils.widgets.text("storage_credential","dbstoragefinancestoragetoken","Storage Credential")
dbutils.widgets.text("catalog_name",       "finance_dev",                "UC Catalog")
dbutils.widgets.text("env",                "dev",                        "Environment (dev/prod)")


In [0]:
# ---------------------------------------------------------------------------
# Input validation — allow-list widget values before any downstream SQL
# interpolation to prevent SQL injection via overridden widget parameters.
# ---------------------------------------------------------------------------
import re

def _validate_identifier(value: str, name: str) -> str:
    """Allow only alphanumeric, underscore, and hyphen characters."""
    if not re.match(r'^[A-Za-z0-9_\-]+$', value.strip()):
        raise ValueError(
            f"Invalid widget value for '{name}': '{value}'. "
            "Only alphanumeric, underscore, and hyphen characters are permitted."
        )
    return value.strip()

STORAGE_ACCOUNT    = _validate_identifier(dbutils.widgets.get("storage_account"),    "storage_account")
STORAGE_CREDENTIAL = _validate_identifier(dbutils.widgets.get("storage_credential"), "storage_credential")
CATALOG_NAME       = _validate_identifier(dbutils.widgets.get("catalog_name"),       "catalog_name")
ENV                = _validate_identifier(dbutils.widgets.get("env"),                "env")

# Schema names
BRONZE_SCHEMA = f"bronze_{ENV}"
SILVER_SCHEMA = f"silver_{ENV}"
GOLD_SCHEMA   = f"gold_{ENV}"

# ABFS root paths
def abfs(container: str) -> str:
    return f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

BRONZE_PATH  = abfs("bronze")
SILVER_PATH  = abfs("silver")
GOLD_PATH    = abfs("gold")
CATALOG_ROOT = abfs(f"finance-{ENV}")

# Fully-qualified table name helper
def fq(schema: str, table: str) -> str:
    return f"{CATALOG_NAME}.{schema}.{table}"

print(f"Catalog  : {CATALOG_NAME}")
print(f"Bronze   : {CATALOG_NAME}.{BRONZE_SCHEMA}  →  {BRONZE_PATH}")
print(f"Silver   : {CATALOG_NAME}.{SILVER_SCHEMA}  →  {SILVER_PATH}")
print(f"Gold     : {CATALOG_NAME}.{GOLD_SCHEMA}    →  {GOLD_PATH}")
print(f"Credential: {STORAGE_CREDENTIAL}")
